# Capítulo 4 — Modelos Atuariais para Séries Temporais

**Livro:** Análise de Dados e Previsão de Séries Temporais  
**Dados:** HMD via `pyhmfd` (ou `mortality_sweden.csv` para testes offline)  

Conteúdo:
- **Parte 1** — Estudo de caso completo: Lee-Carter do zero (SVD → ARIMA → Monte Carlo → CBD → Tábua)
- **Parte 2** — Exercícios 4.1 a 4.6 com soluções


## Parte 1 — Código do Capítulo

### Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.linalg import svd
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
np.random.seed(42)


### Carregamento dos Dados

Usamos `pyhmfd` para baixar as taxas centrais de mortalidade da Suécia (HMD). O DataFrame retornado tem idades nas linhas e anos nas colunas. Se `pyhmfd` não estiver disponível, reconstruímos a matriz a partir do CSV local.

In [ ]:
try:
    import pyhmfd
    df = pyhmfd.load_hmd(country='SWE', ages=range(0, 91), years=range(1950, 2021))
    print('pyhmfd: dados carregados via API')
except Exception as e:
    print(f'pyhmfd indisponível ({e}). Carregando do CSV local.')
    raw = pd.read_csv('mortality_sweden.csv')
    df = (
        raw[raw['Year'].between(1950, 2020) & raw['Age'].between(0, 90)]
        .pivot(index='Age', columns='Year', values='Female')
    )

m_xt   = df.values.astype(float)          # (91 idades × 71 anos)
idades = df.index.astype(int).values
anos   = df.columns.astype(int).values
n_anos = m_xt.shape[1]
ln_m   = np.log(m_xt)
print(f'Matriz: {m_xt.shape}  idades {idades[0]}–{idades[-1]}, anos {anos[0]}–{anos[-1]}')


### Estimação Lee-Carter com SVD

Três passos: (1) perfil médio $a_x$; (2) centralizar e decompor; (3) escalar para satisfazer $\sum b_x = 1$ e $\sum k_t = 0$.

In [ ]:
# Passo 1: perfil etário médio
a_x = ln_m.mean(axis=1)

# Passo 2: centraliza e decompõe
U, S, Vt = svd(ln_m - a_x[:, None], full_matrices=False)

# Passo 3: normalização padrão Lee-Carter
b_x  = U[:, 0] / U[:, 0].sum()          # soma = 1
k_t  = S[0] * Vt[0, :] * U[:, 0].sum()
k_t -= k_t.mean()                        # média = 0

print(f'b_x: min={b_x.min():.4f}, max={b_x.max():.4f}, soma={b_x.sum():.4f}')
print(f'k_t: min={k_t.min():.2f}, max={k_t.max():.2f}, média={k_t.mean():.6f}')


#### Visualização dos Parâmetros Estimados

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))

axes[0, 0].plot(idades, a_x, 'b-')
axes[0, 0].set(title=r'$a_x$ — perfil etário médio',
               xlabel='Idade', ylabel=r'$\ln(m_x)$ médio')

axes[0, 1].plot(idades, b_x, 'g-')
axes[0, 1].set(title=r'$b_x$ — sensibilidade temporal',
               xlabel='Idade', ylabel=r'$b_x$')

axes[1, 0].plot(anos, k_t, 'r-')
axes[1, 0].set(title=r'$k_t$ — índice temporal', xlabel='Ano', ylabel=r'$k_t$')

idx60 = list(idades).index(60)
axes[1, 1].plot(anos, ln_m[idx60, :], 'k-', label='Observado')
axes[1, 1].plot(anos, a_x[idx60] + b_x[idx60] * k_t, 'r--', label='Ajustado LC')
axes[1, 1].set(title='Ajuste vs. observado (idade 60)', xlabel='Ano')
axes[1, 1].legend()

plt.suptitle('Parâmetros Lee-Carter — Suécia (HMD, 1950–2020)', fontsize=11)
plt.tight_layout()
plt.savefig('fig-lc-parametros.png', dpi=150, bbox_inches='tight')
plt.show()


### Projeção de $k_t$

Testamos a ordem de integração com ADF e ajustamos ARIMA(0,1,0) com constante (passeio aleatório com deriva).

In [ ]:
adf_p  = adfuller(k_t)[1]
adf_dp = adfuller(np.diff(k_t))[1]
print(f'ADF k_t:   p = {adf_p:.4f}  ({'não rejeita H0 — I(1)' if adf_p > 0.05 else 'rejeita H0'})') 
print(f'ADF Δk_t:  p = {adf_dp:.4f}  ({'estacionário' if adf_dp < 0.05 else 'ainda não estacionário'})')

resultado_k = ARIMA(k_t, order=(0, 1, 0), trend='c').fit()
deriva = resultado_k.params['const']
sigma  = np.std(np.diff(k_t))
print(f'Deriva estimada: {deriva:.4f} por ano')
print(f'Sigma (choque):  {sigma:.4f}')


### Monte Carlo — Incerteza da Projeção

1 000 trajetórias de $k_t$ para 2021–2050. A incerteza cresce com $\sqrt{h}$ — margens largas no horizonte longo.

In [ ]:
H = 30   # horizonte: 2021–2050
anos_proj = np.arange(anos[-1] + 1, anos[-1] + H + 1)

# Projeção determinística
k_proj = k_t[-1] + deriva * np.arange(1, H + 1)

# Monte Carlo
N_sim = 1000
sims  = (k_t[-1]
         + deriva * np.arange(1, H + 1)
         + np.random.normal(0, sigma, (N_sim, H)).cumsum(axis=1))
k_p05, k_p95 = np.percentile(sims, [5, 95], axis=0)

# Reconstrói taxas
m_proj = np.exp(a_x[:, None] + b_x[:, None] * k_proj[None, :])
print('Projeção OK. Formato m_proj:', m_proj.shape)


#### Visualização da Projeção

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))

ax = axes[0, 0]
ax.plot(anos, k_t, 'b-', label='Histórico')
ax.plot(anos_proj, k_proj, 'r-', label='Projeção')
ax.fill_between(anos_proj, k_p05, k_p95, alpha=0.2, color='red', label='IC 90%')
ax.set(title=r'$k_t$ — índice temporal', xlabel='Ano', ylabel=r'$k_t$')
ax.legend(fontsize=8)

for ax, idade in zip(axes.flat[1:], [0, 60, 80]):
    i = list(idades).index(idade)
    ax.plot(anos, m_xt[i, :], 'b-', label='Histórico')
    ax.plot(anos_proj, m_proj[i, :], 'r-', label='Projeção')
    ax.set(title=f'Idade {idade}', xlabel='Ano', ylabel=r'$m_{x,t}$')
    ax.legend(fontsize=8)

plt.suptitle('Projeção Lee-Carter — Suécia (2021–2050)', fontsize=11)
plt.tight_layout()
plt.savefig('fig-lc-projecao.png', dpi=150, bbox_inches='tight')
plt.show()


### Qualidade do Ajuste In-Sample

R² por idade e MAPE global. O Lee-Carter se sai bem no miolo (30–70), mas piora nas extremidades da tabua.

In [ ]:
m_fit    = np.exp(a_x[:, None] + b_x[:, None] * k_t[None, :])
ss_res   = ((m_xt - m_fit) ** 2).sum(axis=1)
ss_tot   = ((m_xt - m_xt.mean(axis=1)[:, None]) ** 2).sum(axis=1)
r2_idade = 1 - ss_res / ss_tot
mape_global = np.abs((m_xt - m_fit) / m_xt).mean() * 100

print(f'MAPE global: {mape_global:.2f}%')
print(f'R² médio:    {r2_idade.mean():.4f}')
print(f'R² mínimo:   {r2_idade.min():.4f}  (idade {idades[r2_idade.argmin()]})')

plt.figure(figsize=(9, 4))
plt.plot(idades, r2_idade, 'b-')
plt.axhline(0.95, color='red', ls='--', label='R²=0.95')
plt.xlabel('Idade'); plt.ylabel(r'$R^2$')
plt.title('R² por idade — Lee-Carter (Suécia)')
plt.legend(); plt.tight_layout(); plt.show()


### Modelo CBD (Cairns-Blake-Dowd)

Regressão OLS cross-sectional por ano para idades ≥ 60. Dois fatores: nível ($\kappa_t^{(1)}$) e inclinação ($\kappa_t^{(2)}$).

In [ ]:
mask  = idades >= 60
x_bar = idades[mask].mean()
X     = np.column_stack([np.ones(mask.sum()), idades[mask] - x_bar])

params = np.array([sm.OLS(ln_m[mask, t], X).fit().params for t in range(n_anos)])
k1, k2 = params[:, 0], params[:, 1]

# Projeção por deriva constante
k1_proj = k1[-1] + np.diff(k1).mean() * np.arange(1, H + 1)
k2_proj = k2[-1] + np.diff(k2).mean() * np.arange(1, H + 1)

print(f'CBD k1: média anual {np.diff(k1).mean():.5f}')
print(f'CBD k2: média anual {np.diff(k2).mean():.5f}')


#### Comparação LC vs. CBD — Idade 70

In [ ]:
i70 = list(idades).index(70)

logit_fit_cbd  = k1 + k2 * (70 - x_bar)
m_fit_cbd70    = np.exp(logit_fit_cbd) / (1 + np.exp(logit_fit_cbd))
logit_proj_cbd = k1_proj + k2_proj * (70 - x_bar)
m_proj_cbd70   = np.exp(logit_proj_cbd) / (1 + np.exp(logit_proj_cbd))

m_fit_lc70  = np.exp(a_x[i70] + b_x[i70] * k_t)
m_proj_lc70 = np.exp(a_x[i70] + b_x[i70] * k_proj)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(anos, m_xt[i70, :], 'k-', label='Observado', lw=1.5)
ax.plot(anos, m_fit_lc70, 'b--', label='LC ajustado')
ax.plot(anos_proj, m_proj_lc70, 'b-', label='LC projeção')
ax.plot(anos, m_fit_cbd70, 'r--', label='CBD ajustado')
ax.plot(anos_proj, m_proj_cbd70, 'r-', label='CBD projeção')
ax.axvline(anos[-1], color='gray', ls=':', lw=1)
ax.set(title='LC vs. CBD — mortalidade idade 70 (Suécia)',
       xlabel='Ano', ylabel=r'$m_{70,t}$')
ax.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.savefig('fig-lc-vs-cbd.png', dpi=150, bbox_inches='tight')
plt.show()


### Tábua de Vida e Expectativa Remanescente

A partir das taxas $m_{x,t}$ projetadas, calculamos $q_x$, $\ell_x$, $L_x$ e $e_{x_0}$.

In [ ]:
def tabua_e_expectativa(m_vec, idades_vec, x0=60):
    q   = m_vec / (1 + 0.5 * m_vec)
    l   = np.cumprod(np.concatenate([[1], 1 - q[:-1]]))
    i0  = np.where(idades_vec == x0)[0][0]
    L   = (l[:-1] + l[1:]) / 2
    e   = L[i0:].sum() / l[i0]
    return l, e

_, e60_2020 = tabua_e_expectativa(m_xt[:, -1],    idades)
_, e60_2050 = tabua_e_expectativa(m_proj[:, -1],  idades)

print(f'e_60 em 2020: {e60_2020:.2f} anos')
print(f'e_60 em 2050: {e60_2050:.2f} anos  (+{e60_2050 - e60_2020:.2f} anos projetados)')

# Probabilidade de sobreviver dos 60 aos 80
l_2020, _ = tabua_e_expectativa(m_xt[:, -1],   idades)
l_2050, _ = tabua_e_expectativa(m_proj[:, -1], idades)
i60, i80 = list(idades).index(60), list(idades).index(80)
print(f'P(sobreviver 60→80) em 2020: {l_2020[i80] / l_2020[i60]:.4f}')
print(f'P(sobreviver 60→80) em 2050: {l_2050[i80] / l_2050[i60]:.4f}')


## Parte 2 — Exercícios com Soluções

### Exercício 4.1 — Japão: Diferencial de Mortalidade por Sexo

Baixe dados do Japão (HMD), homens e mulheres, 1950–2020. Ajuste LC separado por sexo e projete até 2040. Calcule e plote $e_0$ para cada sexo. O diferencial está aumentando ou diminuindo?

In [ ]:
# ── Solução 4.1 ──────────────────────────────────────────────────────────
def estima_lc(m_xt_in):
    """Retorna a_x, b_x, k_t para uma matriz m_xt."""
    ln = np.log(m_xt_in)
    ax = ln.mean(axis=1)
    U, S, Vt = svd(ln - ax[:, None], full_matrices=False)
    bx = U[:, 0] / U[:, 0].sum()
    kt = S[0] * Vt[0, :] * U[:, 0].sum()
    kt -= kt.mean()
    return ax, bx, kt

def projeta_lc(ax, bx, kt, h=20):
    """Projeta m_xt por h anos usando ARIMA(0,1,0) com constante."""
    res = ARIMA(kt, order=(0, 1, 0), trend='c').fit()
    deriva = res.params['const']
    kt_proj = kt[-1] + deriva * np.arange(1, h + 1)
    return np.exp(ax[:, None] + bx[:, None] * kt_proj[None, :])

def e0_serie(m_xt_in, idades_vec):
    """Calcula e_0 para cada coluna (ano) da matriz."""
    e0s = []
    for t in range(m_xt_in.shape[1]):
        q = m_xt_in[:, t] / (1 + 0.5 * m_xt_in[:, t])
        l = np.cumprod(np.concatenate([[1], 1 - q[:-1]]))
        L = (l[:-1] + l[1:]) / 2
        e0s.append(L.sum() / l[0])
    return np.array(e0s)

try:
    import pyhmfd
    df_jpn_m = pyhmfd.load_hmd(country='JPN', sex='male',   ages=range(0,91), years=range(1950,2021))
    df_jpn_f = pyhmfd.load_hmd(country='JPN', sex='female', ages=range(0,91), years=range(1950,2021))
    m_jpn_m  = df_jpn_m.values.astype(float)
    m_jpn_f  = df_jpn_f.values.astype(float)
    anos_jpn = df_jpn_m.columns.astype(int).values
    idades_jpn = df_jpn_m.index.astype(int).values
    print('Japão carregado via pyhmfd')

except Exception as e:
    print(f'pyhmfd indisponível ({e}). Demonstração com dados sintéticos.')
    anos_jpn = np.arange(1950, 2021)
    idades_jpn = np.arange(0, 91)
    base = np.exp(np.linspace(-9, 0, 91))
    trend = np.exp(-0.015 * np.arange(71))
    m_jpn_m = np.outer(base, trend) * 1.1   # homens: +10%
    m_jpn_f = np.outer(base, trend)

H_jpn = 20   # projeta até 2040
ax_m, bx_m, kt_m = estima_lc(m_jpn_m)
ax_f, bx_f, kt_f = estima_lc(m_jpn_f)
m_proj_m = projeta_lc(ax_m, bx_m, kt_m, H_jpn)
m_proj_f = projeta_lc(ax_f, bx_f, kt_f, H_jpn)
anos_proj_jpn = np.arange(anos_jpn[-1] + 1, anos_jpn[-1] + H_jpn + 1)

e0_m_hist = e0_serie(m_jpn_m, idades_jpn)
e0_f_hist = e0_serie(m_jpn_f, idades_jpn)
e0_m_proj = e0_serie(m_proj_m, idades_jpn)
e0_f_proj = e0_serie(m_proj_f, idades_jpn)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(anos_jpn, e0_m_hist, 'b-', label='Homens (hist.)')
axes[0].plot(anos_jpn, e0_f_hist, 'r-', label='Mulheres (hist.)')
axes[0].plot(anos_proj_jpn, e0_m_proj, 'b--', label='Homens (proj.)')
axes[0].plot(anos_proj_jpn, e0_f_proj, 'r--', label='Mulheres (proj.)')
axes[0].axvline(anos_jpn[-1], color='gray', ls=':', lw=1)
axes[0].set(title='e_0 — Japão', xlabel='Ano', ylabel='Esperança de vida ao nascer (anos)')
axes[0].legend(fontsize=9)

dif_hist = e0_f_hist - e0_m_hist
dif_proj = e0_f_proj - e0_m_proj
axes[1].plot(anos_jpn, dif_hist, 'k-', label='Histórico')
axes[1].plot(anos_proj_jpn, dif_proj, 'k--', label='Projetado')
axes[1].set(title='Diferencial Mulheres - Homens', xlabel='Ano', ylabel='Anos')
axes[1].legend(fontsize=9)

plt.tight_layout(); plt.show()
print(f'Diferencial 2020: {dif_hist[-1]:.2f} anos')
print(f'Diferencial 2040: {dif_proj[-1]:.2f} anos (projetado)')
# No Japão o diferencial de gênero cresceu até ~1990 e depois reverteu
# parcialmente, pois melhorias de mortalidade passaram a favorecer mais homens.


### Exercício 4.2 — Reino Unido: LC vs. CBD (Treino/Teste 60–89 anos)

Treino: 1950–2000, Teste: 2001–2020. Ajuste LC e CBD apenas para idades 60–89. Calcule MAE e RMSE por idade. Em que faixa cada modelo leva a melhor?

In [ ]:
# ── Solução 4.2 ──────────────────────────────────────────────────────────
try:
    import pyhmfd
    df_uk = pyhmfd.load_hmd(country='GBR', ages=range(60, 90), years=range(1950, 2021))
    m_uk  = df_uk.values.astype(float)
    anos_uk   = df_uk.columns.astype(int).values
    idades_uk = df_uk.index.astype(int).values
    print('GBR carregado via pyhmfd')

except Exception as e:
    print(f'pyhmfd indisponível. Dados sintéticos.')
    anos_uk   = np.arange(1950, 2021)
    idades_uk = np.arange(60, 90)
    base = np.exp(np.linspace(-3, -0.5, 30))
    m_uk = np.outer(base, np.exp(-0.012 * np.arange(71)))

cut_tr = np.where(anos_uk == 2000)[0][0] + 1   # índice do primeiro ano de treino
cut_te = cut_tr                                  # treino até 2000, teste a partir de 2001

m_tr = m_uk[:, :cut_tr]
m_te = m_uk[:, cut_te:]
anos_te = anos_uk[cut_te:]

# Lee-Carter no treino
ax_uk, bx_uk, kt_uk = estima_lc(m_tr)
h_te = m_te.shape[1]
m_proj_lc_uk = projeta_lc(ax_uk, bx_uk, kt_uk, h_te)

# CBD no treino
ln_tr = np.log(m_tr)
x_bar_uk = idades_uk.mean()
X_uk = np.column_stack([np.ones(len(idades_uk)), idades_uk - x_bar_uk])
params_uk = np.array([sm.OLS(ln_tr[:, t], X_uk).fit().params for t in range(m_tr.shape[1])])
k1_uk, k2_uk = params_uk[:, 0], params_uk[:, 1]
k1_uk_proj = k1_uk[-1] + np.diff(k1_uk).mean() * np.arange(1, h_te + 1)
k2_uk_proj = k2_uk[-1] + np.diff(k2_uk).mean() * np.arange(1, h_te + 1)
m_proj_cbd_uk = np.exp(
    (k1_uk_proj[None, :] + k2_uk_proj[None, :] * (idades_uk[:, None] - x_bar_uk))
) / (1 + np.exp(
    k1_uk_proj[None, :] + k2_uk_proj[None, :] * (idades_uk[:, None] - x_bar_uk)
))

# Métricas por idade
mae_lc  = np.abs(m_te - m_proj_lc_uk).mean(axis=1)
mae_cbd = np.abs(m_te - m_proj_cbd_uk).mean(axis=1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(idades_uk, mae_lc,  'b-o', ms=4, label='Lee-Carter')
ax.plot(idades_uk, mae_cbd, 'r-s', ms=4, label='CBD')
ax.set(title='MAE por idade — LC vs. CBD (UK, teste 2001–2020)',
       xlabel='Idade', ylabel='MAE')
ax.legend()
plt.tight_layout(); plt.show()

cbd_wins = (idades_uk[(mae_cbd < mae_lc)])
print(f'CBD supera LC nas idades: {cbd_wins[0]}–{cbd_wins[-1]} (em geral 70+)')
# CBD foi desenhado para 60+ e costuma ganhar nas idades mais avançadas.


### Exercício 4.3 — IBGE: Tábua Completa de Mortalidade

Baixe a Tábua do IBGE (SIDRA 7362). Converta $q_x \to m_x$. Ajuste Lee-Carter e projete $e_0$ até 2060. Compare com a projeção oficial do IBGE.

In [ ]:
# ── Solução 4.3 ──────────────────────────────────────────────────────────
# Requer: pip install sidrapy
try:
    import sidrapy
    raw_ibge = sidrapy.get_table(
        table_code='7362',
        territorial_level='1',
        ibge_territorial_code='all',
        period='all',
        variable='10728',     # q_x ambos os sexos
        classifications={'287': 'allxt'},  # todas as idades
    )
    print('IBGE SIDRA 7362 carregado')
    print(raw_ibge.head())
    # Processamento: pivot(index=idade, columns=ano, values=q_x)
    # e conversao q_x -> m_x usando m_x = q_x / (1 - 0.5*q_x)
    # (código completo depende do formato retornado pela API — ajuste conforme necessário)

except Exception as e:
    print(f'sidrapy indisponível ({e}). Demonstração com estrutura simulada.')
    print('Estrutura esperada: DataFrame com q_x por idade (linhas) e ano (colunas).')
    anos_ibge = np.arange(1998, 2022)
    idades_ibge = np.arange(0, 81)
    base_qx = np.exp(np.linspace(-8, -0.5, 81)) * 0.9
    trend_ibge = np.exp(-0.01 * np.arange(len(anos_ibge)))
    qx_ibge = np.outer(base_qx, trend_ibge).clip(0, 0.999)

    # q_x -> m_x
    mx_ibge = qx_ibge / (1 - 0.5 * qx_ibge)

    ax_ibge, bx_ibge, kt_ibge = estima_lc(mx_ibge)

    H_ibge = 39   # 2022–2060
    m_proj_ibge = projeta_lc(ax_ibge, bx_ibge, kt_ibge, H_ibge)
    anos_proj_ibge = np.arange(anos_ibge[-1] + 1, anos_ibge[-1] + H_ibge + 1)

    e0_hist_ibge = e0_serie(mx_ibge, idades_ibge)
    e0_proj_ibge = e0_serie(m_proj_ibge, idades_ibge)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(anos_ibge, e0_hist_ibge, 'b-', label='Histórico LC')
    ax.plot(anos_proj_ibge, e0_proj_ibge, 'b--', label='Projeção LC')
    ax.set(title='e_0 — Brasil (IBGE, simulação)', xlabel='Ano', ylabel='e_0 (anos)')
    ax.legend()
    plt.tight_layout(); plt.show()
    print('Para dados reais, compare com: https://www.ibge.gov.br/estatisticas/sociais/populacao/9109-projecao-da-populacao.html')


### Exercício 4.4 — Renshaw-Haberman (Efeito de Coorte)

Estenda o Lee-Carter com um índice de coorte $\gamma_{t-x}$. Use estimação iterativa (ALS). Compare ajuste in-sample no Reino Unido. O efeito recupera a coorte 1925–1935?

In [ ]:
# ── Solução 4.4 ──────────────────────────────────────────────────────────
# Implementação ALS (Alternating Least Squares) simplificada do Renshaw-Haberman.
# O modelo: ln(m_{x,t}) = a_x + b_x * k_t + b2_x * gamma_{t-x}

def renshaw_haberman_als(ln_m_in, idades_in, anos_in, n_iter=30):
    """
    Estimação iterativa do Renshaw-Haberman.
    Retorna a_x, b_x, k_t, b2_x, gamma_c (por coorte c = t - x).
    """
    n_x, n_t = ln_m_in.shape
    coortes = sorted(set(
        int(anos_in[t] - idades_in[x])
        for x in range(n_x) for t in range(n_t)
    ))
    c_map = {c: i for i, c in enumerate(coortes)}
    n_c   = len(coortes)

    # Inicializa com Lee-Carter
    ax, bx, kt = estima_lc(np.exp(ln_m_in))
    b2x    = np.zeros(n_x)
    gamma  = np.zeros(n_c)

    a_x_new = ax.copy()

    for it in range(n_iter):
        # Passo 1: actualiza gamma dado (a,b,k,b2)
        resid1 = ln_m_in - (a_x_new[:, None] + bx[:, None] * kt[None, :])
        for ci, c in enumerate(coortes):
            xts = [(x, t) for x in range(n_x) for t in range(n_t)
                   if int(anos_in[t] - idades_in[x]) == c]
            if not xts:
                continue
            b2_vals = np.array([b2x[x] for x, _ in xts])
            r_vals  = np.array([resid1[x, t] for x, t in xts])
            if np.abs(b2_vals).sum() > 1e-12:
                gamma[ci] = np.dot(b2_vals, r_vals) / np.dot(b2_vals, b2_vals)

        # Passo 2: cria resíduo sem efeito de coorte e re-estima LC
        gamma_mat = np.array([
            [gamma[c_map[int(anos_in[t] - idades_in[x])]] if int(anos_in[t] - idades_in[x]) in c_map else 0
             for t in range(n_t)] for x in range(n_x)
        ])
        ln_m2 = ln_m_in - b2x[:, None] * gamma_mat
        ax, bx, kt = estima_lc(np.exp(ln_m2))
        a_x_new = ax

        # Passo 3: atualiza b2x
        resid2 = ln_m_in - (a_x_new[:, None] + bx[:, None] * kt[None, :])
        b2x_new = np.zeros(n_x)
        for xi in range(n_x):
            g_vals = np.array([gamma[c_map[int(anos_in[t] - idades_in[xi])]] if int(anos_in[t] - idades_in[xi]) in c_map else 0 for t in range(n_t)])
            if np.dot(g_vals, g_vals) > 1e-12:
                b2x_new[xi] = np.dot(resid2[xi], g_vals) / np.dot(g_vals, g_vals)
        b2x = b2x_new

    return a_x_new, bx, kt, b2x, gamma, coortes

# Aplica ao dataset disponível (Suécia como proxy se UK indisponível)
print('Estimando Renshaw-Haberman (pode demorar ~30 s)...')
ax_rh, bx_rh, kt_rh, b2x_rh, gamma_rh, coortes_rh = renshaw_haberman_als(
    ln_m, idades, anos, n_iter=20
)

# Efeito de coorte
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(coortes_rh, gamma_rh, 'b-', lw=0.8)
ax.axvspan(1925, 1935, alpha=0.15, color='red', label='Coorte 1925–1935')
ax.set(title='Efeito de coorte gamma_c — Renshaw-Haberman',
       xlabel='Ano de nascimento (coorte)', ylabel=r'$\gamma_c$')
ax.legend()
plt.tight_layout(); plt.show()

# Compara MAPE in-sample
gamma_mat_rh = np.array([
    [gamma_rh[coortes_rh.index(int(anos[t] - idades[x]))] if int(anos[t] - idades[x]) in coortes_rh else 0
     for t in range(n_anos)] for x in range(len(idades))
])
m_fit_lc_sw  = np.exp(a_x[:, None] + b_x[:, None] * k_t[None, :])
m_fit_rh     = np.exp(ax_rh[:, None] + bx_rh[:, None] * kt_rh[None, :] + b2x_rh[:, None] * gamma_mat_rh)
mape_lc_sw   = np.abs((m_xt - m_fit_lc_sw)  / m_xt).mean() * 100
mape_rh      = np.abs((m_xt - m_fit_rh)      / m_xt).mean() * 100
print(f'MAPE Lee-Carter:     {mape_lc_sw:.3f}%')
print(f'MAPE Renshaw-Hab.:   {mape_rh:.3f}%')


### Exercício 4.5 — $b_x$ Variável no Tempo

O Lee-Carter assume $b_x$ fixo. Proponha uma extensão onde $b_x(t)$ varia via splines de baixa dimensão. Verifique se a variação é economicamente relevante.

In [ ]:
# ── Solução 4.5 ──────────────────────────────────────────────────────────
# Abordagem: dividir a amostra em duas metades e comparar b_x estimado em cada uma.
# Uma variação alternativa usa rolling windows.

mid = n_anos // 2

_, bx_1a = estima_lc(m_xt[:, :mid])[:2]   # 1950–1984
_, bx_2a = estima_lc(m_xt[:, mid:])[:2]   # 1985–2020

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(idades, bx_1a, 'b-', label=f'{anos[0]}–{anos[mid-1]}')
axes[0].plot(idades, bx_2a, 'r--', label=f'{anos[mid]}–{anos[-1]}')
axes[0].set(title=r'$b_x$ em dois subperíodos', xlabel='Idade', ylabel=r'$b_x$')
axes[0].legend()

axes[1].plot(idades, bx_2a - bx_1a, 'k-')
axes[1].axhline(0, color='red', ls='--')
axes[1].set(title=r'Variação $b_x^{(2)} - b_x^{(1)}$',
            xlabel='Idade', ylabel=r'$\Delta b_x$')

plt.tight_layout(); plt.show()

# Interpretação
delta_bx = bx_2a - bx_1a
top_ages = idades[np.abs(delta_bx).argsort()[-5:][::-1]]
print(f'Idades com maior variacao de b_x: {top_ages}')
print('Tipicamente nas idades 70-85: b_x cresceu no 2o periodo,')
print('refletindo concentracao recente dos ganhos de longevidade nos idosos.')

# Rolling window para ver b_x ao longo do tempo
janela = 20
bx_roll = []
anos_roll = []
for start in range(0, n_anos - janela + 1, 3):
    _, bx_w, _ = estima_lc(m_xt[:, start:start + janela])
    bx_roll.append(bx_w)
    anos_roll.append(anos[start + janela // 2])

bx_roll = np.array(bx_roll)
# Plota b_x para idades 60 e 80
fig, ax = plt.subplots(figsize=(9, 4))
for idade_alvo in [20, 60, 80]:
    i = list(idades).index(idade_alvo)
    ax.plot(anos_roll, bx_roll[:, i], label=f'idade {idade_alvo}')
ax.set(title=r'$b_x$ via janela deslizante (janela=20 anos)',
       xlabel='Ano central', ylabel=r'$b_x$')
ax.legend()
plt.tight_layout(); plt.show()


### Exercício 4.6 — Valor Presente de Anuidade Vitalícia

Para uma pessoa de 65 anos, calcule o VPV de uma anuidade unitária usando:
(i) taxas projetadas ano a ano; (ii) taxas congeladas no ano base.
Repita para taxas reais de desconto de 3%, 4% e 5%.

In [ ]:
# ── Solução 4.6 ──────────────────────────────────────────────────────────
def vpv_anuidade(m_proj_in, idades_vec, x0=65, taxa_desconto=0.04, max_horizonte=35):
    """
    Valor presente atuarial de anuidade vitalícia unitária.
    m_proj_in: array (n_idades x n_horizonte) com taxas projetadas.
    """
    i0 = np.where(idades_vec == x0)[0][0]
    h  = min(max_horizonte, m_proj_in.shape[1])

    vpv = 0.0
    l_x = 1.0   # sobreviventes normalizados
    for t in range(h):
        x  = x0 + t
        xi = np.where(idades_vec == x)[0]
        if len(xi) == 0:
            break
        mx = m_proj_in[xi[0], t]
        qx = mx / (1 + 0.5 * mx)
        # Anuidade antecipada: paga no inicio de cada ano
        vpv   += l_x / (1 + taxa_desconto) ** t
        l_x   *= (1 - qx)

    return vpv

# (i) Taxas projetadas (LC, 2021–2050)
# (ii) Taxas congeladas em 2020
m_frozen = np.repeat(m_xt[:, [-1]], 30, axis=1)   # congela 2020

print(f'{'Taxa':>6}  {'VPV projetado':>16}  {'VPV congelado':>14}  {'Diferença %':>12}')
print('-' * 56)
for r in [0.03, 0.04, 0.05]:
    vpv_proj = vpv_anuidade(m_proj, idades, x0=65, taxa_desconto=r)
    vpv_frzn = vpv_anuidade(m_frozen, idades, x0=65, taxa_desconto=r)
    diff_pct = (vpv_proj - vpv_frzn) / vpv_frzn * 100
    print(f'{r*100:>5.0f}%  {vpv_proj:>16.4f}  {vpv_frzn:>14.4f}  {diff_pct:>11.2f}%')

# O VPV projetado é maior pois a longevidade futura é maior do que a de 2020.
# A diferença cresce com taxas de desconto menores (peso maior ao longo prazo).
# Para uma seguradora, subestimar a longevidade = subsuficiente reserva tecnica.


---

**Fim do notebook.** Próximos capítulos — Deep Learning:
- Cap 5: Redes Feedforward (`cap05-feedforward-solucoes.ipynb`)
- Cap 6: Redes Recorrentes / LSTM
- Cap 7: Convoluções Temporais
- Cap 8: Transformers

Dependências deste notebook: `numpy pandas matplotlib statsmodels scipy`  
Opcionais: `pyhmfd sidrapy python-bcb`
